In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
from pathlib import Path
from torch.utils.data import DataLoader

join = os.path.join
import torch

from skimage import io, transform
import torch.nn.functional as F

from pytorch_ood.detector import Mahalanobis
from pytorch_ood.utils import OODMetrics

import monai
from monai.metrics import DiceMetric

from PIL import Image

from peft import LoraConfig, get_peft_model

In [4]:
PROJECT_ROOT=Path("/home/jovyan/thesis_project")
PROJECT_FM_THESIS= PROJECT_ROOT / "FM_thesis"

DATA_ROOT = Path("/home/jovyan/thesis_project/datasets/INbreast/INbreast Release 1.0/MassCases")

MEDSAM_IMAGE_SIZE=(1024,1024)
VERSAMAMMO_IMAGE_SIZE=(224,224)
#CSV_DATA_PATH= DATA_ROOT / "CLaM-Annot-metadata.csv"

MEDSAM_ROOT= PROJECT_FM_THESIS / "MedSAM"
VERSAMAMMO_ROOT = PROJECT_FM_THESIS / "VersaMammo"
VERSAMAMMO_DETECTION_ROOT = VERSAMAMMO_ROOT / "downstream" / "Detection"

WEIGHTS_PATH= PROJECT_ROOT / "models"
MEDSAM_ZEROSHOT = WEIGHTS_PATH / "medsam_vit_b.pth"
MEDSAM_LORA_ZGT= WEIGHTS_PATH / "medsam_LoRA_ZGT_from_gt_masks_fold4.pth"
VERSAMAMMO_ZEROSHOT = WEIGHTS_PATH / "VersaMammo_pretrained" / "VersaMammo (Enb5).pth"

OUTPUT_FEATURES=PROJECT_FM_THESIS / "feature_space_inspection"
OUTPUT_FEATURES.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"


In [6]:
for path in [PROJECT_FM_THESIS, MEDSAM_ROOT, VERSAMAMMO_DETECTION_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))



from utils.dep_injection_util import (
        BaseDataset,
        NpzLoader,
        PngLoader,
        ConnectedComponentsBBoxFromMask
)

from MedSAM.segment_anything import sam_model_registry
from MedSAM.utils.medSAM_architecture import MedSAM, MedSAMPreprocess
from VersaMammo.models.image_encoder import build_vit_model


xFormers is not available (SwiGLU)
xFormers is not available (Attention)
xFormers is not available (Block)


In [7]:


sam_model = sam_model_registry["vit_b"](checkpoint=MEDSAM_ZEROSHOT)

model = MedSAM(
    image_encoder=sam_model.image_encoder,
    mask_decoder=sam_model.mask_decoder,
    prompt_encoder=sam_model.prompt_encoder,
).to(DEVICE)


config = LoraConfig(
    r=8,
    lora_alpha=8,
    target_modules=["qkv"],
    lora_dropout=0.1,
    bias="none"
)
model = get_peft_model(model, config)


checkpoint = torch.load(MEDSAM_LORA_ZGT, map_location=DEVICE)
model.load_state_dict(checkpoint["model"])

model.eval()

PeftModel(
  (base_model): LoraModel(
    (model): MedSAM(
      (image_encoder): ImageEncoderViT(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
        )
        (blocks): ModuleList(
          (0-11): 12 x Block(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (attn): Attention(
              (qkv): lora.Linear(
                (base_layer): Linear(in_features=768, out_features=2304, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()


In [5]:

def collect_ZGT_files(root, metadata_df):
    root = Path(root) if isinstance(root, str) else root

    samples = []
    for row in metadata_df.itertuples():
        image_file= root / Path(row.ImagePath)
        mask_file=  root / Path(row.ROIPath) if Path(row.ROIPath).name != "None" else None

        samples.append({
            "image_path": image_file,
            "mask_path": mask_file,
        })  

    return samples

In [8]:
png_loader = PngLoader(dtype=np.uint8)

npz_loader = NpzLoader(dtype=None)

bbox_generator = ConnectedComponentsBBoxFromMask(
    annotation_threshold=0.5,
    allow_empty_mask=True,
)


# id_dataset = BaseDataset(
#     root=DATA_ROOT,
#     format_loader=loader,
#     file_collector=collect_ZGT_files,
#     collector_kwargs={"metadata_df": df_masses},
#     bbox_generator=bbox_generator,
#     transforms=MedSAMPreprocess(MEDSAM_IMAGE_SIZE)
# )





ood_INbreast_MedSAM_dataset = BaseDataset(
    root=DATA_ROOT,
    format_loader=npz_loader,
    file_collector=None,
    bbox_generator=bbox_generator,
    transforms=MedSAMPreprocess(MEDSAM_IMAGE_SIZE),
    label=-1
)

In [13]:
def pytorchood_mahalanobis_cov(z, y, eps=1e-6):
    device = z.device
    classes = y.unique()
    n_classes = len(classes)

    mu = torch.zeros((n_classes, z.shape[-1]), device=device)
    cov = torch.zeros((z.shape[-1], z.shape[-1]), device=device)

    for clazz in range(n_classes):
        idx = y.eq(clazz)
        zs = z[idx]

        mu[clazz] = zs.mean(dim=0)
        centered = zs - mu[clazz]

        cov += centered.T @ centered

    cov += torch.eye(cov.shape[0], device=device) * eps
    precision = torch.linalg.inv(cov)

    return mu, cov, precision

In [14]:
#MDS code block for fitting on pre-saved features


Z_id = torch.load(
    OUTPUT_FEATURES / "medSAM_LoRA_ZGT_allmasses",
    map_location=DEVICE
)

Y_id=torch.zeros(Z_id.shape[0])

detector = Mahalanobis(model=None)
detector.fit_features(Z_id, Y_id)


mu_manual, cov_manual, precision_manual = pytorchood_mahalanobis_cov(Z_id, Y_id)

print(torch.allclose(detector.mu, mu_manual, rtol=1e-4, atol=1e-5))
print(torch.allclose(detector.cov, cov_manual, rtol=1e-4, atol=1e-5))
print(torch.allclose(detector.precision, precision_manual, rtol=1e-4, atol=1e-5))

No device given. Will use 'cuda:0'.


True
True
True


In [ ]:
# Pass ID data through encoder and fit Mahalanobis distance metric
feats_list_id = []
labels_list_id = []



Z_id = torch.load(
    OUTPUT_FEATURES / "medSAM_LoRA_ZGT_allmasses",
    map_location=DEVICE
)


with torch.no_grad():
    for sample in id_dataset:
        imgs = sample["image"].to(DEVICE).unsqueeze(0)   #unsqueeze only if this is not using the batch loader
        y = sample["label"].to(DEVICE)

        z = model.image_encoder(imgs)              # [B, C, H, W] feature maps
        z = z.mean(dim=(2, 3))                     # [B, C] global average pool

        feats_list_id.append(z)
        labels_list_id.append(y)

Z_id = torch.cat(feats_list_id, dim=0)
Y_id = torch.stack(labels_list_id)

detector = Mahalanobis(encoder=None)
detector.fit_features(Z_id, Y_id)




In [9]:
torch.save(Z_id.detach().cpu(), OUTPUT_FEATURES / "medSAM_LoRA_ZGT_ID_allmasses")

In [15]:
# Predict on OOD dataset
feats_list = []

with torch.no_grad():
    for sample in ood_INbreast_MedSAM_dataset:
        imgs = sample["image"].to(DEVICE).unsqueeze(0)
        z = model.image_encoder(imgs)
        z = z.mean(dim=(2, 3))
        feats_list.append(z)

Z_ood = torch.cat(feats_list, dim=0)


#torch.save(Z_ood.detach().cpu(), OUTPUT_FEATURES / "medSAM_LoRA_ZGT-ID_INBreast-OOD_masses")

scores_id = detector.predict_features(Z_id)
scores_ood = detector.predict_features(Z_ood)

In [16]:
def pytorchood_mahalanobis_scores(z, mu, precision):
    md_k = []

    for clazz in range(mu.shape[0]):
        centered = z - mu[clazz]
        term = -0.5 * ((centered @ precision) * centered).sum(dim=1)
        md_k.append(term[:, None])

    md_k = torch.cat(md_k, dim=1)

    # PyTorch-OOD convention: larger score = more OOD
    scores = -torch.max(md_k, dim=1).values

    return scores


scores_manual = pytorchood_mahalanobis_scores(Z_ood, detector.mu, detector.precision)
scores_api = detector.predict_features(Z_ood)

print(torch.allclose(scores_manual, scores_api, rtol=1e-4, atol=1e-5))
print(scores_manual.mean(), scores_manual.std())
print(scores_api.mean(), scores_api.std())

True
tensor(4066.3923, device='cuda:0') tensor(1087.0510, device='cuda:0')
tensor(4066.3923, device='cuda:0') tensor(1087.0508, device='cuda:0')


In [17]:
# 1) Make label tensors (same length as scores)
y_id  = torch.zeros(len(scores_id), dtype=torch.long)        # ID label = 0
y_ood = -torch.ones(len(scores_ood), dtype=torch.long)       # OOD label = -1

# 2) Concatenate
scores = torch.cat([scores_id.detach().cpu(), scores_ood.detach().cpu()], dim=0)
y_true = torch.cat([y_id, y_ood], dim=0)

# 3) Compute metrics
metrics = OODMetrics()
metrics.update(scores, y_true)
result = metrics.compute()

print(result)

{'AUROC': 1.0, 'AUTC': 0.244190976023674, 'AUPR-IN': 1.0, 'AUPR-OUT': 1.0, 'FPR95TPR': 0.0}


In [21]:
from sklearn.metrics import roc_auc_score
from pytorch_ood.detector import Mahalanobis

detector = Mahalanobis(model=None)
detector.fit_features(Z_id, Y_id)

s_id = detector.predict_features(Z_id)
s_ood = detector.predict_features(Z_ood)

print("API ID:", s_id.mean(), s_id.std(), s_id.min(), s_id.max())
print("API OOD:", s_ood.mean(), s_ood.std(), s_ood.min(), s_ood.max())

y_true = torch.cat([
    torch.zeros(len(s_id)),
    torch.ones(len(s_ood)),
]).cpu().numpy()

scores_api = torch.cat([s_id, s_ood]).detach().cpu().numpy()

print("AUROC API raw:", roc_auc_score(y_true, scores_api))
print("AUROC API negated:", roc_auc_score(y_true, -scores_api))

No device given. Will use 'cuda:0'.


API ID: tensor(0.4447, device='cuda:0') tensor(0.0820, device='cuda:0') tensor(0.2392, device='cuda:0') tensor(0.4889, device='cuda:0')
API OOD: tensor(4066.3923, device='cuda:0') tensor(1087.0508, device='cuda:0') tensor(2873.1030, device='cuda:0') tensor(7940.7729, device='cuda:0')
AUROC API raw: 1.0
AUROC API negated: 0.0


## VersaMammo encoder features for OOD detection


In [22]:
class VersaMammoEncoderPreprocess:
    """Preprocess images for the VersaMammo ViT-B/14 encoder.

    The encoder is built with img_size=224 and returns a C x 16 x 16 patch feature map.
    We keep this transform image-only because the OOD pipeline only needs encoder features.
    """

    def __init__(self, image_size=(224, 224), mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)):
        self.image_size = image_size
        self.mean = torch.tensor(mean, dtype=torch.float32).view(3, 1, 1)
        self.std = torch.tensor(std, dtype=torch.float32).view(3, 1, 1)

    def __call__(self, sample):
        image = sample["image"]
        image = np.asarray(image)

        if image.ndim == 3:
            if image.shape[-1] == 1:
                image = image[..., 0]
            elif image.shape[-1] == 3:
                image = image.mean(axis=-1)
            elif image.shape[0] in (1, 3):
                image = image.mean(axis=0)
            else:
                raise ValueError(f"Unexpected image shape for VersaMammo preprocessing: {image.shape}")

        image = image.astype(np.float32)
        image_min = image.min()
        image_max = image.max()
        image = (image - image_min) / max(image_max - image_min, 1e-8)

        image = torch.from_numpy(image).float().unsqueeze(0).repeat(3, 1, 1)
        image = F.interpolate(
            image.unsqueeze(0),
            size=self.image_size,
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)
        image = (image - self.mean) / self.std

        sample["image"] = image
        return sample


def pool_encoder_features(features):
    if isinstance(features, (tuple, list)):
        features = features[0]
    if features.ndim == 4:
        return features.mean(dim=(2, 3))
    if features.ndim == 3:
        return features.mean(dim=1)
    if features.ndim == 2:
        return features
    return features.flatten(start_dim=1)


def extract_encoder_features(dataset, encoder, device):
    feats_list = []
    labels_list = []

    encoder.eval()
    with torch.no_grad():
        for sample in dataset:
            imgs = sample["image"].to(device).unsqueeze(0)
            labels_list.append(sample["label"].to(device))

            features = encoder(imgs)
            features = pool_encoder_features(features)
            feats_list.append(features)

    return torch.cat(feats_list, dim=0), torch.stack(labels_list)


In [23]:
if not VERSAMAMMO_ZEROSHOT.exists():
    raise FileNotFoundError(
        f"VersaMammo encoder checkpoint not found: {VERSAMAMMO_ZEROSHOT}. "
        "Update VERSAMAMMO_ZEROSHOT to the teacher_checkpoint_*.pth file you want to use."
    )

versamammo_encoder, versamammo_num_features = build_vit_model(
    arch="vit_base",
    pretrained=True,
    pretrained_path=str(VERSAMAMMO_ZEROSHOT),
)
versamammo_encoder = versamammo_encoder.to(DEVICE).eval()
print(f"Loaded VersaMammo ViT encoder with {versamammo_num_features} features.")


Parameter module.image_encoder._conv_stem.weight not found in the model state_dict.
Parameter module.image_encoder._bn0.weight not found in the model state_dict.
Parameter module.image_encoder._bn0.bias not found in the model state_dict.
Parameter module.image_encoder._bn0.running_mean not found in the model state_dict.
Parameter module.image_encoder._bn0.running_var not found in the model state_dict.
Parameter module.image_encoder._bn0.num_batches_tracked not found in the model state_dict.
Parameter module.image_encoder._blocks.0._depthwise_conv.weight not found in the model state_dict.
Parameter module.image_encoder._blocks.0._bn1.weight not found in the model state_dict.
Parameter module.image_encoder._blocks.0._bn1.bias not found in the model state_dict.
Parameter module.image_encoder._blocks.0._bn1.running_mean not found in the model state_dict.
Parameter module.image_encoder._blocks.0._bn1.running_var not found in the model state_dict.
Parameter module.image_encoder._blocks.0._bn

In [24]:
png_loader = PngLoader(dtype=np.uint8)

npz_loader = NpzLoader(dtype=None)

bbox_generator = ConnectedComponentsBBoxFromMask(
    annotation_threshold=0.5,
    allow_empty_mask=True,
)


# versamammo_id_dataset = BaseDataset(
#     root=DATA_ROOT,
#     format_loader=loader,
#     file_collector=collect_ZGT_files,
#     collector_kwargs={"metadata_df": df_masses},
#     bbox_generator=bbox_generator,
#     transforms=VersaMammoEncoderPreprocess(VERSAMAMMO_IMAGE_SIZE),
# )



ood_INbreast_VersaMammo_dataset = BaseDataset(
    root=DATA_ROOT,
    format_loader=npz_loader,
    file_collector=None,
    bbox_generator=bbox_generator,
    transforms=VersaMammoEncoderPreprocess(VERSAMAMMO_IMAGE_SIZE),
    label=-1,
)



In [26]:
#preloaded features from versamammo id zgt

Z_id_versamammo = torch.load(
    OUTPUT_FEATURES / "VersaMammo_ZGT_allmasses",
    map_location=DEVICE
)

Y_id_versamammo=torch.zeros(Z_id_versamammo.shape[0])

versamammo_detector = Mahalanobis(model=None)
versamammo_detector.fit_features(Z_id_versamammo, Y_id_versamammo)

No device given. Will use 'cuda:0'.


In [ ]:
# Z_id_versamammo, Y_id_versamammo = extract_encoder_features(
#     versamammo_id_dataset,
#     versamammo_encoder,
#     DEVICE,
# )

# torch.save(Z_id_versamammo.detach().cpu(), OUTPUT_FEATURES / "VersaMammo_ZGT_ID_allmasses")

# versamammo_detector = Mahalanobis(encoder=None)
# versamammo_detector.fit_features(Z_id_versamammo, Y_id_versamammo)


In [27]:
Z_ood_versamammo, _ = extract_encoder_features(
    ood_INbreast_VersaMammo_dataset,
    versamammo_encoder,
    DEVICE,
)

#torch.save(Z_ood_versamammo.detach().cpu(), OUTPUT_FEATURES / "VersaMammo_INBreast_masses")

scores_id_versamammo = versamammo_detector.predict_features(Z_id_versamammo)
scores_ood_versamammo = versamammo_detector.predict_features(Z_ood_versamammo)


In [30]:
y_id_versamammo = torch.zeros(len(scores_id_versamammo), dtype=torch.long)
y_ood_versamammo = -torch.ones(len(scores_ood_versamammo), dtype=torch.long)

scores_versamammo = torch.cat(
    [scores_id_versamammo.detach().cpu(), scores_ood_versamammo.detach().cpu()],
    dim=0,
)
y_true_versamammo = torch.cat([y_id_versamammo, y_ood_versamammo], dim=0)

versamammo_metrics = OODMetrics()
versamammo_metrics.update(scores_versamammo, y_true_versamammo)
versamammo_result = versamammo_metrics.compute()

print(versamammo_result)


{'AUROC': 0.0, 'AUTC': 0.686153769493103, 'AUPR-IN': 0.3851965665817261, 'AUPR-OUT': 0.20224718749523163, 'FPR95TPR': 1.0}
